# Phase 1 Final Comparator Reconciliation

This notebook is an orchestration and audit surface only. Durable scientific logic lives in `src/` and is exercised through `python -m src.cli phase1_final_comparator_reconciliation`.

Scientific integrity boundary:

- This step reconciles two already generated output packages: the feature-matrix final comparator runner and the final A2d covariance/tangent runner.
- It must not rerun models, change logits, recompute metrics, fabricate missing files, or promote smoke artifacts.
- The A2d missing-output blocker may be cleared only at the artifact-manifest level after A2d output manifest and runtime leakage log pass.
- Claims remain closed because controls, calibration, influence and final reporting are still not reconciled.
- BA/ECE/Brier values remain diagnostics only and must not be interpreted as efficacy evidence.

In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from __future__ import annotations

import getpass
import hashlib
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
RUN_UNITTESTS = True

def run(cmd, cwd=None, check=True):
    printable = ['<redacted>' if 'Authorization:' in str(part) else str(part) for part in cmd]
    print('$', ' '.join(printable))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

if not REPO_DIR.exists():
    token = getpass.getpass('Enter GitHub token for private repo, or leave blank for public clone: ')
    if token:
        header = 'Authorization: Basic ' + __import__('base64').b64encode(f'x-access-token:{token}'.encode()).decode()
        run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
    else:
        run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

print('Repo:', REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)

if RUN_UNITTESTS:
    unit = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, text=True)
    if unit.returncode != 0:
        raise subprocess.CalledProcessError(unit.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])

In [ ]:
# Cell 2 - Select reviewed source artifacts and keep launch disabled by default.

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'

FEATURE_MATRIX_COMPARATOR_RUN = None  # Optional explicit Path(...); defaults to latest artifact.
FINAL_A2D_RUN = None  # Optional explicit Path(...); defaults to latest artifact.

RUN_FINAL_COMPARATOR_RECONCILIATION = False
REQUIRED_ACK = 'I understand this reconciliation is claim-closed and does not prove Phase 1 efficacy'
MANUAL_LAUNCH_ACK = ''

def latest_run(root: Path) -> Path:
    latest = root / 'latest.txt'
    if latest.exists():
        candidate = Path(latest.read_text().strip())
        if candidate.exists():
            return candidate
    runs = sorted([path for path in root.iterdir() if path.is_dir()])
    if not runs:
        raise FileNotFoundError(f'No runs under {root}')
    return runs[-1]

if FEATURE_MATRIX_COMPARATOR_RUN is None:
    FEATURE_MATRIX_COMPARATOR_RUN = latest_run(DRIVE_ROOT / 'artifacts/phase1_final_comparator_runner')
if FINAL_A2D_RUN is None:
    FINAL_A2D_RUN = latest_run(DRIVE_ROOT / 'artifacts/phase1_final_a2d_runner')

PLAN_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_comparator_reconciliation_plan'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_comparator_reconciliation'

print('Feature-matrix comparator run:', FEATURE_MATRIX_COMPARATOR_RUN)
print('Final A2d run:', FINAL_A2D_RUN)
print('Run flag:', RUN_FINAL_COMPARATOR_RECONCILIATION)

In [ ]:
# Cell 3 - Validate prereg identity and source package boundaries before planning.

def read_json(path: Path):
    return json.loads(path.read_text())

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

fm_summary = read_json(FEATURE_MATRIX_COMPARATOR_RUN / 'phase1_final_comparator_runner_summary.json')
a2d_summary = read_json(FINAL_A2D_RUN / 'phase1_final_a2d_runner_summary.json')
a2d_patch = read_json(FINAL_A2D_RUN / 'phase1_final_a2d_completeness_patch.json')

required_resolved = {
    'A2d_riemannian_not_executable_from_final_feature_matrix',
    'A2d_riemannian_final_covariance_runner_missing',
}
preflight_blockers = []
if 'A2d_riemannian' not in fm_summary.get('blocked_comparators', []):
    preflight_blockers.append('feature_matrix_runner_did_not_record_a2d_blocker')
if a2d_summary.get('a2d_final_output_present') is not True:
    preflight_blockers.append('a2d_final_output_missing')
if a2d_summary.get('runtime_leakage_passed') is not True:
    preflight_blockers.append('a2d_runtime_leakage_not_passed')
if not required_resolved.issubset(set(a2d_patch.get('resolved_blockers_for_downstream_reconciliation', []))):
    preflight_blockers.append('a2d_patch_missing_required_resolved_blockers')
if fm_summary.get('claim_ready') is not False or a2d_summary.get('claim_ready') is not False:
    preflight_blockers.append('source_package_claim_not_closed')

print('Feature-matrix runner status:', fm_summary.get('status'))
print('Feature-matrix completed comparators:', fm_summary.get('completed_comparators'))
print('Feature-matrix blocked comparators:', fm_summary.get('blocked_comparators'))
print('A2d runner status:', a2d_summary.get('status'))
print('A2d output present:', a2d_summary.get('a2d_final_output_present'))
print('A2d runtime leakage passed:', a2d_summary.get('runtime_leakage_passed'))
print('Preflight blockers:', preflight_blockers)

In [ ]:
# Cell 4 - Record reconciliation plan/manual hold artifact.

timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir = PLAN_ROOT / timestamp
plan_dir.mkdir(parents=True, exist_ok=False)
plan = {
    'status': 'phase1_final_comparator_reconciliation_plan_recorded',
    'created_utc': timestamp,
    'run_flag': RUN_FINAL_COMPARATOR_RECONCILIATION,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'preflight_blockers': preflight_blockers,
    'prereg_bundle': str(PREREG_BUNDLE),
    'locked_prereg_identity_hash': actual_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_sha256,
    'feature_matrix_comparator_run': str(FEATURE_MATRIX_COMPARATOR_RUN),
    'final_a2d_run': str(FINAL_A2D_RUN),
    'output_root': str(OUTPUT_ROOT),
    'scope': 'artifact-manifest reconciliation only; no model rerun and no claim opening',
    'not_ok_to_claim': [
        'decoder efficacy',
        'A2d efficacy',
        'A3/A4 efficacy',
        'A4 superiority',
        'privileged-transfer efficacy',
        'full Phase 1 neural comparator performance',
    ],
}
(plan_dir / 'phase1_final_comparator_reconciliation_plan.json').write_text(json.dumps(plan, indent=2) + '\n')
print(json.dumps(plan, indent=2))

In [ ]:
# Cell 5 - Manual hold or launch reconciliation CLI.

if preflight_blockers:
    raise RuntimeError(f'Preflight blockers must be resolved before reconciliation: {preflight_blockers}')
elif not RUN_FINAL_COMPARATOR_RECONCILIATION:
    hold = {
        'status': 'phase1_final_comparator_reconciliation_manual_hold',
        'plan_dir': str(plan_dir),
        'run_flag': RUN_FINAL_COMPARATOR_RECONCILIATION,
        'required_ack': REQUIRED_ACK,
        'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    }
    print(json.dumps(hold, indent=2))
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not reconcile final comparators without explicit non-claim acknowledgement.')
else:
    cmd = [
        'python', '-m', 'src.cli', 'phase1_final_comparator_reconciliation',
        '--profile', 't4_safe',
        '--config', str(PREREG_BUNDLE),
        '--feature-matrix-comparator-run', str(FEATURE_MATRIX_COMPARATOR_RUN),
        '--final-a2d-run', str(FINAL_A2D_RUN),
        '--output-root', str(OUTPUT_ROOT),
        '--reconciliation-config', 'configs/phase1/final_comparator_reconciliation.json',
    ]
    run(cmd, cwd=REPO_DIR)
    print('Final comparator reconciliation command completed. Review artifacts before downstream governance.')

In [ ]:
# Cell 6 - Inspect latest reconciliation output if execution ran.

latest = OUTPUT_ROOT / 'latest.txt'
print('latest exists:', latest.exists())
if latest.exists():
    run_dir = Path(latest.read_text().strip())
    print('run_dir:', run_dir)
    required_files = [
        'phase1_final_comparator_reconciliation_inputs.json',
        'phase1_final_comparator_reconciliation_summary.json',
        'phase1_final_comparator_reconciliation_report.md',
        'phase1_final_comparator_reconciliation_source_links.json',
        'phase1_final_comparator_reconciliation_input_validation.json',
        'phase1_final_comparator_reconciled_completeness_table.json',
        'phase1_final_comparator_reconciled_runtime_leakage_audit.json',
        'phase1_final_comparator_reconciled_output_manifest_index.json',
        'phase1_final_comparator_reconciled_claim_state.json',
    ]
    for name in required_files:
        print(name, 'exists =', (run_dir / name).exists())
    summary = read_json(run_dir / 'phase1_final_comparator_reconciliation_summary.json')
    completeness = read_json(run_dir / 'phase1_final_comparator_reconciled_completeness_table.json')
    leakage = read_json(run_dir / 'phase1_final_comparator_reconciled_runtime_leakage_audit.json')
    print(json.dumps({
        'status': summary.get('status'),
        'completed_comparators': summary.get('completed_comparators'),
        'blocked_comparators': summary.get('blocked_comparators'),
        'all_final_comparator_outputs_present': summary.get('all_final_comparator_outputs_present'),
        'runtime_logs_audited_for_all_required': summary.get('runtime_comparator_logs_audited_for_all_required_comparators'),
        'claim_ready': summary.get('claim_ready'),
        'claim_blockers': summary.get('claim_blockers'),
        'completeness_status': completeness.get('status'),
        'runtime_leakage_status': leakage.get('status'),
    }, indent=2))
else:
    print('No reconciliation run was launched in this pass.')

In [ ]:
# Cell 7 - Assertions for launched reconciliation output.

if RUN_FINAL_COMPARATOR_RECONCILIATION:
    assert latest.exists(), 'Reconciliation latest pointer missing.'
    assert summary.get('status') == 'phase1_final_comparator_reconciliation_complete_claim_closed', summary.get('claim_blockers')
    assert summary.get('all_final_comparator_outputs_present') is True
    assert summary.get('runtime_comparator_logs_audited_for_all_required_comparators') is True
    assert summary.get('claim_ready') is False
    assert summary.get('headline_phase1_claim_open') is False
    assert summary.get('full_phase1_claim_bearing_run_allowed') is False
    assert summary.get('smoke_artifacts_promoted') is False
    assert 'final_comparator_outputs_incomplete' not in summary.get('claim_blockers', [])
    assert 'A2d_riemannian_final_covariance_runner_missing' not in summary.get('claim_blockers', [])
    assert 'controls_calibration_influence_reporting_missing' in summary.get('claim_blockers', [])
    assert all(row.get('status') == 'completed_claim_closed' for row in completeness.get('rows', []))
    assert leakage.get('outer_test_subject_used_for_any_fit') is False
    print('Reconciliation assertions passed. Claims remain closed.')
else:
    print('Assertions skipped because manual flag is False.')

In [ ]:
# Cell 8 - Write a non-claim review note after launched reconciliation.

if RUN_FINAL_COMPARATOR_RECONCILIATION and latest.exists():
    review_note = {
        'status': 'phase1_final_comparator_reconciliation_review_pass_non_claim',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'reviewed_run': str(run_dir),
        'scope': 'final comparator artifact-manifest reconciliation only',
        'checks_passed': [
            'feature_matrix_runner_linked',
            'final_a2d_runner_linked',
            'all_six_comparator_output_manifests_present',
            'all_six_logits_present',
            'all_six_subject_metrics_present',
            'all_six_runtime_leakage_logs_present',
            'a2d_missing_output_blocker_resolved_at_artifact_level',
            'claim_ready_false',
            'headline_phase1_claim_open_false',
            'controls_calibration_influence_reporting_still_blocking_claims',
        ],
        'completed_comparators': summary.get('completed_comparators'),
        'blocked_comparators': summary.get('blocked_comparators'),
        'claim_blockers': summary.get('claim_blockers'),
        'metrics_interpretation': {
            'allowed': 'Implementation and artifact-completeness diagnostics only.',
            'not_allowed': 'Do not use comparator metrics as Phase 1 efficacy evidence until controls, calibration, influence and reporting pass under governance.',
        },
        'not_ok_to_claim': [
            'decoder efficacy',
            'A2d efficacy',
            'A3/A4 efficacy',
            'A4 superiority',
            'privileged-transfer efficacy',
            'full Phase 1 neural comparator performance',
        ],
    }
    review_path = run_dir / 'phase1_final_comparator_reconciliation_review_note.json'
    review_path.write_text(json.dumps(review_note, indent=2) + '\n')
    print('Review note written:', review_path)
    print(json.dumps(review_note, indent=2))
else:
    print('Review note skipped because reconciliation was not launched.')

In [ ]:
# Cell 9 - Closeout.

print('================ Phase 1 Final Comparator Reconciliation Closeout ================')
print('Notebook fix marker: phase1_final_comparator_reconciliation_v1_20260422')
print('Plan source of truth:', plan_dir)
print('Run flag:', RUN_FINAL_COMPARATOR_RECONCILIATION)
print('Feature-matrix comparator run:', FEATURE_MATRIX_COMPARATOR_RUN)
print('Final A2d run:', FINAL_A2D_RUN)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
if RUN_FINAL_COMPARATOR_RECONCILIATION and latest.exists():
    print('Latest reconciliation output:', run_dir)
    print('All final comparator outputs present:', summary.get('all_final_comparator_outputs_present'))
    print('Completed comparators:', summary.get('completed_comparators'))
    print('Blocked comparators:', summary.get('blocked_comparators'))
    print('Claim blockers:', summary.get('claim_blockers'))
    print('CHECK REQUIRED: Review reconciled completeness table, runtime leakage audit and claim state before downstream controls/calibration/influence/reporting.')
else:
    print('HELD: Reconciliation was not launched because manual flag is False.')
    print('NEXT: review the plan, then rerun only with explicit non-claim acknowledgement if appropriate.')
print('\nNOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')